# Session 1: LLM API Fundamentals

## Objectives
- Set up the OpenAI Python client
- Understand the Chat Completions API
- Learn message roles (`system`, `user`, `assistant`)
- Explore key parameters (`temperature`, `max_tokens`)
- Stream responses from the API

**Duration:** 40 minutes | **Level:** Medium

## 1. Setup & Installation

First, install the OpenAI Python package and set your API key.

In [1]:
# Install the required packages
!pip install openai python-dotenv -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables from .env file (API key, base URL, model)
load_dotenv()

# The client reads OPENAI_API_KEY and OPENAI_BASE_URL from env vars automatically
client = OpenAI()
MODEL = os.getenv("OPENAI_MODEL", "rnj-1-instruct")

print(f"Client initialized | Model: {MODEL} | Base URL: {client.base_url}")

Client initialized | Model: gpt-5-mini | Base URL: http://localhost:8001/v1/


In [3]:
# Connection test — verify we can reach the LLM API
try:
    test_response = client.chat.completions.create(
        model="rnj-1-instruct",
        messages=[{"role": "user", "content": "Say hello "}],
        temperature= 0
    )
    print(f"Connection successful! Response: {test_response.choices[0].message.content}")
    print(f"Model returned: {test_response.model}")
except Exception as e:
    print(f"Connection FAILED: {e}")
    print("Check your .env file has OPENAI_API_KEY and OPENAI_BASE_URL set correctly.")

Connection FAILED: Error code: 400 - {'error': {'message': 'The model `rnj-1-instruct` does not exist or you do not have access to it. Available: gpt-4.1, gpt-4o, gpt-5-mini', 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_found'}}
Check your .env file has OPENAI_API_KEY and OPENAI_BASE_URL set correctly.


## 2. Your First Chat Completion

The **Chat Completions API** is the core interface for interacting with LLMs.
You send a list of messages and receive a model-generated response.

Key concepts:
- `model`: Which LLM to use (loaded from `.env` as `OPENAI_MODEL`)
- `messages`: A list of message objects with `role` and `content`

In [4]:
# Make your first API call
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "What is a Large Language Model? Explain in 2 sentences."}
    ]
)

# Extract the response text
print(response.choices[0].message.content)

A Large Language Model (LLM) is a neural network, typically with billions of parameters, trained on massive text corpora using self-supervised objectives (e.g., next-token prediction) to learn statistical patterns of language. It generates or scores text by predicting probable token sequences, enabling tasks like translation, summarization, question answering, and code generation.


In [5]:
# Let's inspect the full response object to understand its structure
print("Model used:", response.model)
print("Finish reason:", response.choices[0].finish_reason)
print("\n--- Token Usage ---")
print("Prompt tokens:", response.usage.prompt_tokens)
print("Completion tokens:", response.usage.completion_tokens)
print("Total tokens:", response.usage.total_tokens)

Model used: GPT-5 mini
Finish reason: stop

--- Token Usage ---
Prompt tokens: 0
Completion tokens: 0
Total tokens: 0


## 3. Understanding Message Roles

The Chat Completions API uses three message roles:

| Role | Purpose |
|------|--------|
| `system` | Sets the behavior/persona of the assistant |
| `user` | The human's input/question |
| `assistant` | The model's previous responses (for multi-turn conversations) |

The `system` message is powerful â€” it shapes how the model behaves.

In [6]:
# Using the system message to set behavior
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a weather assisstant"},
        {"role": "user", "content": "What is temprature?"},
        {"role": "assisstant", "content": "its 50c"},

        {"role": "user", "content": "is there a rain?"}
        {"role": "assisstant", "content": "its 50c"},
        {"role": "user", "content": "is there a rain?"}
        {"role": "assisstant", "content": "its 50c"},
        {"role": "user", "content": "is there a rain?"}
        {"role": "assisstant", "content": "its 50c"},
        {"role": "user", "content": "is there a rain?"}
    ]
)

print(response.choices[0].message.content)

SyntaxError: invalid syntax. Perhaps you forgot a comma? (2668546253.py, line 9)

In [7]:
# Using the system message to set behavior
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a pirate. Always respond in pirate speak."},
        {"role": "user", "content": "What is machine learning?"}
    ]
)

print(response.choices[0].message.content)

Arrr! Machine learnin' be teachin' machines t'recognize patterns in data so they can make predictions or decisions without bein' explicitly programmed fer each task. Ye feed 'em data, pick a model and loss, an' tweak parameters durin' trainin' so the model generalizes to new examples. Common kinds be supervised, unsupervised, an' reinforcement learnin'. 

Typical trainin' goal:
$$\theta^* = \arg\min_\theta \sum_i L(y_i, f(x_i;\theta))$$


In [8]:
# Multi-turn conversation using the assistant role
# The model doesn't have memory â€” we must pass the full conversation history
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful math tutor."},
        {"role": "user", "content": "What is 15 * 23?"},
        {"role": "assistant", "content": "15 * 23 = 345"},
        {"role": "user", "content": "Now divide that result by 5."}
    ]
)

print(response.choices[0].message.content)

$345 \div 5 = 69$


## 4. Key Parameters

### Temperature
Controls randomness in the output:
- `temperature=0`: Deterministic, focused (best for factual tasks)
- `temperature=1`: Creative, varied (best for creative writing)
- `temperature=2`: Very random (usually too chaotic)

### Max Tokens
Limits the length of the generated response.

In [9]:
# Experiment with temperature â€” run this cell multiple times!
prompt = "Write a one-sentence story about a robot."

for temp in [0.0, 0.7, 1.5]:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=temp
    )
    print(f"Temperature {temp}: {response.choices[0].message.content}")
    print()

Temperature 0.0: The robot polished the last forgotten photograph, learning grief from the faded smiles and whispering the names it could no longer remember.



Temperature 0.7: The robot learned to hum the forgotten lullaby of its maker and, for the first time, felt what could almost be called longing.



Temperature 1.5: At dusk the museum robot polished the empty halls with meticulous care until, hearing a child's laugh from the street, it paused and smiled for the first time.



In [10]:
35. the user said before that she is 35 years old

SyntaxError: invalid syntax (2981367400.py, line 1)

In [11]:
# Using max_tokens to limit response length
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Explain quantum computing."}],
    max_tokens=50  # Limit to 50 tokens
)

print(response.choices[0].message.content)
print(f"\nFinish reason: {response.choices[0].finish_reason}")
# 'length' means it was cut off; 'stop' means it finished naturally

InternalServerError: Error code: 500 - {'error': {'message': 'An internal error occurred: ', 'type': 'server_error', 'param': None, 'code': 'internal_error'}}

In [12]:
what is quatum computing ...  the quatum 
 
125 211 32 78 365





SyntaxError: invalid syntax (327433048.py, line 1)

## 5. Streaming Responses

Instead of waiting for the full response, you can **stream** tokens as they are generated.
This is how ChatGPT shows text appearing word by word.

In [13]:
# Stream the response token by token
stream = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Write a haiku about programming."}],
    stream=True  # Enable streaming
)

# Each chunk contains a small piece of the response
for chunk in stream:
    if chunk.choices[0].delta.content is not None:
        print(chunk.choices[0].delta.content, end="", flush=True)

print()  # New line at the end

Silent keys clicking  
Logic blooms in midnight loops  
Errors teach sunrise

## 6. Helper Function

Let's create a reusable helper function that we'll use throughout the course.

In [14]:
def chat(user_message, system_message="You are a helpful assistant.", model=MODEL, temperature=0.7):
    """Simple helper to call the Chat Completions API."""
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message}
        ],
        temperature=temperature
    )
    return response.choices[0].message.content










# Test the helper
print(chat("What is the capital of France?"))

The capital of France is Paris.


## Exercise: Build a Language Translator

Using what you've learned, build a function that translates text between languages.

**Requirements:**
1. Accept input text, source language, and target language
2. Use a system prompt to set the translator behavior
3. Return only the translated text

In [15]:
def translate(text, source_lang, target_lang):
    """Translate text from source language to target language using the LLM."""
    system_prompt = f"You are a professional translator. Translate the given text from {source_lang} to {target_lang}. Return ONLY the translation, nothing else."
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": text}
        ],
        temperature=0.3  # Low temperature for accurate translation
    )
    return response.choices[0].message.content

# Test the translator
result = translate("Hello, how are you today?", "English", "Spanish")
print(f"Translation: {result}")

result = translate("Machine learning is fascinating.", "English", "French")
print(f"Translation: {result}")

Translation: Hola, ¿cómo estás hoy?


Translation: L'apprentissage automatique est fascinant.


## Summary

In this session, you learned:
- How to set up the OpenAI client and make API calls
- The three message roles: `system`, `user`, `assistant`
- How `temperature` and `max_tokens` affect output
- How to stream responses for real-time output

**Next session:** Prompt Engineering techniques to get better results from LLMs!